In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes.network import deepNN
from neuro_bes.preprocessing import profile_transform
import read_from_external_file
from neuro_bes.postprocessing import evaluation


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, max_error
import h5py
import pandas as pd

In [ ]:
# tf.config.threading.set_inter_op_parallelism_threads(32)
# tf.config.threading.set_intra_op_parallelism_threads(32)

In [ ]:
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [ ]:
#r_coord,emission,density=read_synthetic_static_data("static synthetic data/Na_beam/Na_statdataset.hdf5")

r_coord,emission,density=read_from_external_file.read_w7x_experimental('/home/molnarbalazs/data/training_data.hdf5')
threshold=15

#r_coord,emission,density=read_hesel('../data/BES_ML_modelling/HESEL data/',kind='fast')
#r_coord=r_coord[0]
#threshold=1e22

#r_coord,emission,density=read_hesel('../data/BES_ML_modelling/HESEL data/',kind='slow')
#r_coord=r_coord[0]
#threshold=1e22

#r_coord,emission,density=read_asdex_experimental('../data/BES_ML_modelling/ASDEX data/dataset.h5')

In [ ]:
snr=50000000.0
emission=emission*(1.0+1.0/snr*np.random.randn(emission.shape[0],emission.shape[1]))

In [ ]:
mask = np.max(density, axis=1) > threshold
density_dense = density[mask]
emission_dense = emission[mask]
mask = np.max(density, axis=1) < threshold
density_opaque = density[mask]
emission_opaque = emission[mask]

In [ ]:
for i in (np.random.rand(100)*emission.shape[0]).astype(int):
    plt.plot(emission[i])

In [ ]:
for i in (np.random.rand(10)*emission_opaque.shape[0]).astype(int):
    plt.plot(emission_opaque[i])

In [ ]:
for i in (np.random.rand(100)*density.shape[0]).astype(int):
    plt.plot(density[i])

In [ ]:
for i in (np.random.rand(100)*density_opaque.shape[0]).astype(int):
    plt.plot(density_opaque[i])

In [ ]:
plt.plot(np.cumsum(emission,axis=1)[0])

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

X=emission
Y=density

n_samples = X.shape[0]
n_features = X.shape[1]

# Split into training and testing
split_idx = int(0.8 * n_samples)
X_train, X_test = X[:split_idx], X[split_idx:]
Y_train, Y_test = Y[:split_idx], Y[split_idx:]

train_size=split_idx
indices = np.arange(split_idx)
np.random.shuffle(indices)
X_train = X_train[indices[:train_size]]
Y_train = Y_train[indices[:train_size]]

In [ ]:
x_scaler=profile_transform.GlobalMaxScaler()
X_train_scaled=x_scaler.fit_transform(X_train)
X_test_scaled=x_scaler.transform(X_test)
y_scaler=profile_transform.GlobalMaxScaler()
Y_train_scaled=y_scaler.fit_transform(Y_train)
Y_test_scaled=y_scaler.transform(Y_test)

In [ ]:
y_train_max = np.max(Y_train, axis=1)
hist, bin_edges = np.histogram(y_train_max, bins=10)
# Avoid zero counts for inverse
hist = np.where(hist == 0, 1, hist)    # Assign each sample to a bin
bin_indices = np.digitize(y_train_max, bin_edges[:-1], right=True)
# Map each sample to its bin's count
bin_counts = hist[bin_indices - 1]
# Inverse proportional weights
sample_weights = 1.0 / bin_counts
sample_weights = sample_weights.astype(np.float32)
# Normalize weights to mean 1
sample_weights /= np.mean(sample_weights)

In [ ]:
sample_weights.shape

In [ ]:
plt.hist(y_train_max, bins=20)

In [ ]:
plt.hist(y_train_max, bins=20)

In [ ]:
plt.scatter(y_train_max,np.clip(sample_weights, 0, 10))

In [ ]:
# from tensorflow.python.profiler import profiler_v2 as profiler
# profiler.start('logs/')

In [ ]:
n_features

In [ ]:
model=deepNN.make_CNN(data_length=n_features, num_layers=3, filters_per_layer=32, kernel_size_per_layer=15, dropout_rate=0.01, use_batchnorm=False)
#model=deepNN.make_MLP(data_length=n_features, num_layers=6, units_per_layer=128, activations='relu',dropout_rate=0.2)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='mse', metrics=['mape'])
model,history = deepNN.train(model, X_train_scaled, Y_train_scaled, epochs=10, batch_size=256)
model.summary()

In [ ]:
# profiler.stop()

In [ ]:
# !tensorboard --logdir logs/

In [ ]:
Y_test.shape

In [ ]:
mask = np.max(Y_test, axis=1) < threshold
X_test_scaled_opaque = X_test_scaled[mask]
X_test_opaque=X_test[mask]
Y_test_opaque = Y_test[mask]
#X_test_scaled_opaque = X_test_scaled
#Y_test_opaque = Y_test

In [ ]:
y_pred_scaled = model.predict(X_test_scaled_opaque)
y_pred_scaled=y_pred_scaled.reshape(y_pred_scaled.shape[0], -1)
y_pred = y_scaler.inverse_transform(y_pred_scaled)
y_true = Y_test_opaque
metrics={'loss_function':'mse'}
evaluation.calc_mape_stats(y_true,y_pred,metrics)
evaluation.calc_mase_stats(y_true,y_pred,Y_train,metrics)
evaluation.calc_mre(y_true,y_pred,metrics)
metrics['loss']=history.history['loss']
metrics['val_loss']=history.history['val_loss']

In [ ]:
y_true.shape

In [ ]:
plt.subplot(1, 2, 1)
plt.plot(history.history['val_loss'], label='Validation loss')
plt.plot(history.history['loss'], label='Training loss') 
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation loss')
plt.legend()
plt.grid(True)
plt.ylim(0, history.history['val_loss'][5])  

plt.subplot(1, 2, 2)
plt.plot(history.history['val_mape'], label='Validation MAPE')
plt.plot(history.history['mape'], label='Training MAPE') 
plt.xlabel('Epoch')
plt.ylabel('Mean absolute percentage error')
plt.title('Training and Validation MAPE')
plt.legend()
plt.grid(True)
plt.ylim(0, history.history['val_mape'][0])  
plt.show()

In [ ]:
plt.plot(r_coord,metrics['MRE'])

In [ ]:
mase=np.mean(np.abs(y_true-y_pred),axis=1)/np.mean(np.abs((y_true-np.mean(Y_train,axis=0))),axis=1)

In [ ]:
plt.hist(mase,bins=100, log =True)
#np.save('mase.npy', mase)

In [ ]:
y_pred_worst=y_pred[mase>.2]
y_true_worst=y_true[mase>.2]
y_pred_best=y_pred[mase<.2]
y_true_best=y_true[mase<.2]

In [ ]:
plt.subplot(2,1,1)
plt.plot(y_true_best.T)
plt.subplot(2,1,2)
plt.plot(y_true_worst.T)

In [ ]:
best_mase=np.argmin(mase)
plt.plot(y_pred[best_mase,:])
plt.plot(y_true[best_mase,:])
#np.save('best_mase_y_pred.npy',y_pred[best_mase,:])
#np.save('best_mase_y_true.npy',y_true[best_mase,:])

In [ ]:
best_l2=np.argmin(np.mean((y_pred-y_true)**2,axis=1))
plt.plot(y_pred[best_l2,:])
plt.plot(y_true[best_l2,:])

In [ ]:
worst_mase=np.argmax(mase)
plt.plot(y_pred[worst_mase,:])
plt.plot(y_true[worst_mase,:])
#np.save('worst_mase_y_pred.npy',y_pred[worst_mase,:])
#np.save('worst_mase_y_true.npy',y_true[worst_mase,:])

In [ ]:
worst_l2=np.argmax(np.mean((y_pred-y_true)**2,axis=1))
plt.plot(y_pred[worst_l2,:])
plt.plot(y_true[worst_l2,:])

In [ ]:
worst_l1=np.argmax(np.mean(np.abs(y_pred-y_true),axis=1))
plt.plot(y_pred[worst_l1,:])
plt.plot(y_true[worst_l1,:])

In [ ]:
maximum=max_error(y_pred[0,:],y_true[0,:])
worst_pointwise=0
for i in range(y_pred.shape[0]):
    m=max_error(y_pred[i,:],y_true[i,:])
    if maximum<m:
        maximum=m
        worst_pointwise=i
plt.plot(y_pred[worst_pointwise,:])
plt.plot(y_true[worst_pointwise,:])

In [ ]:
#plt.plot(X_test_scaled_opaque[worst_pointwise,:])
plt.plot(X_test_opaque[worst_pointwise,:])

In [ ]:
def MC_dropout(model, x_test, x_scaler, y_test, y_scaler, MC_dropout_samples=100):
    r = np.arange(len(y_test))
    plt.plot(r,y_test)
    pred_mean=np.zeros(len(y_test))
    pred_sq=np.zeros(len(y_test))
    for i in range(MC_dropout_samples):
        pred=np.squeeze(y_scaler.inverse_transform(model(x_scaler.transform(x_test).reshape(1, -1),training=True)[0]))
        pred_mean=pred_mean+pred/MC_dropout_samples
        pred_sq=pred_sq+pred**2/MC_dropout_samples
    pred_std=np.sqrt(pred_sq-pred_mean**2)
    plt.plot(r,pred_mean)
    plt.fill_between(r,pred_mean-pred_std,pred_mean+pred_std,color="orange",alpha=0.3)
    plt.show()

In [ ]:
MC_dropout(model, X_test[worst_l2,:], x_scaler, Y_test[worst_l2,:], y_scaler, MC_dropout_samples=100)

In [ ]:
def max_l2_difference(X: np.ndarray) -> float:
    # Compute the squared norms of each row
    norms_squared = np.sum(X ** 2, axis=1, keepdims=True)  # shape (n, 1)

    # Compute pairwise squared L2 distances using the identity:
    # ||a - b||^2 = ||a||^2 + ||b||^2 - 2·a·b
    dists_squared = norms_squared + norms_squared.T - 2 * X @ X.T

    return np.unravel_index(np.argmax(dists_squared),dists_squared.shape)

In [ ]:
ind1,ind2=max_l2_difference(Y_test_opaque)
plt.plot(Y_test_opaque[ind1,:])
plt.plot(Y_test_opaque[ind2,:])

In [ ]:
plt.plot(X_test[ind1,:])
plt.plot(X_test[ind2,:])

In [ ]:
plt.plot(y_pred[ind1,:])
plt.plot(y_true[ind1,:])

In [ ]:
plt.plot(y_pred[ind2,:])
plt.plot(y_true[ind2,:])

In [ ]:
df = pd.Series(metrics)
print(df)
#df.to_csv('out_temp.csv')